<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/padron_to_parquet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.colab import drive
import os

# Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Paths
file_path    = '/content/drive/MyDrive/Tesis2026/Padron_ARCA_enero.txt'
output_path  = '/content/drive/MyDrive/Tesis2026/padron_juridicas_enero.parquet'

# Layout según el PDF
anchos = [11, 160, 6, 1, 11, 10, 1, 10, 2, 8]
nombres = [
    "cuit", "denominacion", "actividad", "marca_baja",
    "cuit_reemplazo", "fecha_nacimiento", "sexo",
    "codigo_postal", "provincia", "fecha_fallecimiento"
]

# Personas jurídicas: CUIT que empieza con 30, 33 o 34
PREFIJOS_JURIDICAS = ('30', '33', '34')

try:
    reader = pd.read_fwf(
        file_path,
        widths=anchos,
        names=nombres,
        chunksize=100000,
        engine='python',
        dtype=str,
        header=None,
        encoding='latin-1' # Added encoding to handle special characters
    )

    writer      = None
    total_rows  = 0
    total_jur   = 0

    print("Procesando padrón ARCA...")

    for i, chunk in enumerate(reader):

        # Strip en todas las columnas
        chunk = chunk.apply(lambda col: col.str.strip() if col.dtype == object else col)

        # Filtrar solo personas jurídicas por prefijo de CUIT
        mask = chunk['cuit'].str.startswith(PREFIJOS_JURIDICAS)
        chunk = chunk[mask].copy()

        if chunk.empty:
            total_rows += 100000  # aproximado para el log
            continue

        # Convertir columnas numéricas
        for col in ['cuit', 'actividad', 'cuit_reemplazo', 'provincia']:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce')

        # Rellenar nulos en strings
        for col in ['denominacion', 'marca_baja', 'fecha_nacimiento',
                    'sexo', 'codigo_postal', 'fecha_fallecimiento']:
            chunk[col] = chunk[col].fillna('').astype(str)

        table = pa.Table.from_pandas(chunk, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)

        writer.write_table(table)
        total_rows += len(chunk) + chunk.shape[0]  # aprox
        total_jur  += len(chunk)

        if (i + 1) % 10 == 0:
            print(f"Bloque {i+1} OK. Jurídicas acumuladas: {total_jur:,}")

except Exception as e:
    print(f"\nError: {e}")
finally:
    if writer is not None:
        writer.close()
        print(f"\nArchivo Parquet finalizado.")
        print(f"Total personas jurídicas exportadas: {total_jur:,}")

print("Fin del proceso.")

Mounted at /content/drive
Procesando padrón ARCA...
Bloque 10 OK. Jurídicas acumuladas: 28,233
Bloque 20 OK. Jurídicas acumuladas: 56,454

Archivo Parquet finalizado.
Total personas jurídicas exportadas: 73,332


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.colab import drive
import os

# Mount Google Drive (ensure it's mounted, though it should be from the previous run)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Paths
file_path    = '/content/drive/MyDrive/Tesis2026/Padron_ARCA_enero.txt'
output_path_fisicas  = '/content/drive/MyDrive/Tesis2026/padron_fisicas_enero.parquet'

# Layout según el PDF (same as before)
anchos = [11, 160, 6, 1, 11, 10, 1, 10, 2, 8]
nombres = [
    "cuit", "denominacion", "actividad", "marca_baja",
    "cuit_reemplazo", "fecha_nacimiento", "sexo",
    "codigo_postal", "provincia", "fecha_fallecimiento"
]

# Personas jurídicas: CUIT que empieza con 30, 33 o 34
PREFIJOS_JURIDICAS = ('30', '33', '34')

try:
    reader = pd.read_fwf(
        file_path,
        widths=anchos,
        names=nombres,
        chunksize=100000,
        engine='python',
        dtype=str,
        header=None,
        encoding='latin-1'
    )

    writer      = None
    total_rows  = 0
    total_fisicas = 0

    print("Procesando padrón ARCA para personas físicas...")

    for i, chunk in enumerate(reader):

        # Strip en todas las columnas
        chunk = chunk.apply(lambda col: col.str.strip() if col.dtype == object else col)

        # Filtrar solo personas físicas (CUIT que NO empieza con prefijos de jurídicas)
        mask = ~chunk['cuit'].str.startswith(PREFIJOS_JURIDICAS, na=False)
        chunk = chunk[mask].copy()

        if chunk.empty:
            total_rows += 100000  # aproximado para el log
            continue

        # Convertir columnas numéricas (only CUIT and actividad are relevant for this dataset, others are often null for fisicas)
        for col in ['cuit', 'actividad', 'cuit_reemplazo', 'provincia']:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce')

        # Rellenar nulos en strings
        for col in ['denominacion', 'marca_baja', 'fecha_nacimiento',
                    'sexo', 'codigo_postal', 'fecha_fallecimiento']:
            chunk[col] = chunk[col].fillna('').astype(str)

        table = pa.Table.from_pandas(chunk, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(output_path_fisicas, table.schema)

        writer.write_table(table)
        total_rows += len(chunk) + chunk.shape[0]  # aprox
        total_fisicas  += len(chunk)

        if (i + 1) % 10 == 0:
            print(f"Bloque {i+1} OK. Físicas acumuladas: {total_fisicas:,}")

except Exception as e:
    print(f"\nError: {e}")
finally:
    if writer is not None:
        writer.close()
        print(f"\nArchivo Parquet para personas físicas finalizado.")
        print(f"Total personas físicas exportadas: {total_fisicas:,}")

print("Fin del proceso de personas físicas.")

Procesando padrón ARCA para personas físicas...
Bloque 10 OK. Físicas acumuladas: 971,767
Bloque 20 OK. Físicas acumuladas: 1,943,546
Bloque 30 OK. Físicas acumuladas: 2,915,245
Bloque 40 OK. Físicas acumuladas: 3,887,341
Bloque 50 OK. Físicas acumuladas: 4,858,878
Bloque 60 OK. Físicas acumuladas: 5,830,617
Bloque 70 OK. Físicas acumuladas: 6,802,436
Bloque 80 OK. Físicas acumuladas: 7,774,419

Error: Table schema does not match schema used to create file: 
table:
cuit: int64
denominacion: string
actividad: int64
marca_baja: string
cuit_reemplazo: double
fecha_nacimiento: string
sexo: string
codigo_postal: string
provincia: double
fecha_fallecimiento: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1291 vs. 
file:
cuit: int64
denominacion: string
actividad: int64
marca_baja: string
cuit_reemplazo: double
fecha_nacimiento: string
sexo: string
codigo_postal: string
provincia: int64
fecha_fallecimiento: string
-- schema metadata -

In [ ]:
import pandas as pd
import os

# Ensure Google Drive is mounted, if not already.
# This part assumes previous cells have mounted it, but as a safeguard:
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# The output path for juridical persons from the cell above (UCyjAUl3Jg0o)
output_path_juridicas = '/content/drive/MyDrive/Tesis2026/padron_fisicas_enero.parquet'

# CUIL to search for
cuil_to_search = 20455010366  # It was converted to numeric in the previous step

# Check if the parquet file exists
if not os.path.exists(output_path_juridicas):
    print(f"Error: The file '{output_path_juridicas}' was not found. Please ensure the previous cell executed correctly.")
else:
    try:
        # Read only the 'cuit' column to be memory efficient
        df_juridicas = pd.read_parquet(output_path_juridicas, columns=['cuit'])

        # Check if the CUIL exists in the 'cuit' column
        if cuil_to_search in df_juridicas['cuit'].values:
            print(f"The CUIL {cuil_to_search} *is* present in the physical persons' padron.")
        else:
            print(f"The CUIL {cuil_to_search} is *not* present in the physical persons' padron.")

    except Exception as e:
        print(f"An error occurred while reading the parquet file or searching for the CUIL: {e}")


The CUIL 20455010366 is *not* present in the juridical persons' padron.


In [ ]:
import pandas as pd
import os

# Ensure Google Drive is mounted
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Re-using definitions from earlier cells
file_path = '/content/drive/MyDrive/Tesis2026/Padron_ARCA_enero.txt'
anchos = [11, 160, 6, 1, 11, 10, 1, 10, 2, 8]
nombres = [
    "cuit", "denominacion", "actividad", "marca_baja",
    "cuit_reemplazo", "fecha_nacimiento", "sexo",
    "codigo_postal", "provincia", "fecha_fallecimiento"
]

cuil_to_search_str = '20455010366'
found = False

print(f"Searching for CUIL {cuil_to_search_str} in '{file_path}'...")

if not os.path.exists(file_path):
    print(f"Error: The input file '{file_path}' was not found.")
else:
    try:
        reader = pd.read_fwf(
            file_path,
            widths=anchos,
            names=nombres,
            chunksize=100000, # Read in chunks to handle large file
            engine='python',
            dtype=str, # Read all as strings to prevent parsing issues
            header=None,
            encoding='latin-1'
        )

        for i, chunk in enumerate(reader):
            # Strip whitespace from 'cuit' column
            chunk['cuit'] = chunk['cuit'].str.strip()

            if cuil_to_search_str in chunk['cuit'].values:
                found = True
                print(f"CUIL {cuil_to_search_str} *is* present in the original file {file_path}.")
                break # Stop searching once found

            if (i + 1) % 10 == 0:
                print(f"Checked {i+1} blocks...")

        if not found:
            print(f"CUIL {cuil_to_search_str} is *not* present in the original file {file_path}.")

    except Exception as e:
        print(f"An error occurred while reading the file or searching for the CUIL: {e}")

print("Search complete.")


Searching for CUIL 20455010366 in '/content/drive/MyDrive/Tesis2026/Padron_ARCA_enero.txt'...
Checked 10 blocks...
Checked 20 blocks...
Checked 30 blocks...
Checked 40 blocks...
Checked 50 blocks...
Checked 60 blocks...
Checked 70 blocks...
Checked 80 blocks...
Checked 90 blocks...
Checked 100 blocks...
Checked 110 blocks...
Checked 120 blocks...
Checked 130 blocks...
Checked 140 blocks...
Checked 150 blocks...
Checked 160 blocks...
Checked 170 blocks...
Checked 180 blocks...
Checked 190 blocks...
Checked 200 blocks...
Checked 210 blocks...
Checked 220 blocks...
Checked 230 blocks...
CUIL 20455010366 *is* present in the original file /content/drive/MyDrive/Tesis2026/Padron_ARCA_enero.txt.
Search complete.
